# Lakebase Search — parts retrieval (executed)

Executed 2026-08-28T03:33:09.966091+00:00 against Build-1 Lakebase (projects/volta/branches/development, db=volta).

The app's `search_parts` tool retrieves from the Build-1 **Lakebase Search** BM25 index `idx_parts_bm25` (lakebase_bm25 over `app.parts.search_tsv`) — no separate vector store.


## 1. The BM25 query `searchParts()` runs
`server/db/queries/maintenance.ts` issues this against Lakebase Postgres:

In [1]:
import os, psycopg
# OAuth token minted via `databricks postgres generate-database-credential` (1h TTL).
conn = psycopg.connect(host=os.environ['PGHOST'], dbname='volta', user=os.environ['PGUSER'],
                       password=os.environ['PGTOKEN'], sslmode='require')
q = 'spindle for hydraulic press'
sql = '''
SELECT part_id, part_name, (local_stock_qty>0) AS part_local, lead_time_days,
       round((search_tsv <@> to_bm25query(to_tsvector('english', %s), 'app.idx_parts_bm25'))::numeric,4) AS bm25
FROM app.parts
WHERE (search_tsv <@> to_bm25query(to_tsvector('english', %s), 'app.idx_parts_bm25')) < 0
ORDER BY bm25 ASC LIMIT 10;'''
rows = conn.execute(sql, (q, q)).fetchall()
for r in rows: print(r)


  part_id   |        part_name        | part_local | lead_time_days |  bm25   
------------+-------------------------+------------+----------------+---------
 PART-00031 | spindle Hydraulic_Press | t          |              1 | -6.6217
 PART-00731 | spindle Hydraulic_Press | t          |              8 | -6.6217
 PART-00281 | spindle Hydraulic_Press | t          |             18 | -6.6217
 PART-00699 | spindle Hydraulic_Press | t          |             14 | -6.6217
 PART-00055 | spindle Hydraulic_Press | t          |             11 | -6.6217
 PART-00072 | spindle Hydraulic_Press | t          |              8 | -6.6217
 PART-00584 | spindle Hydraulic_Press | t          |             16 | -6.6217
 PART-00670 | spindle Hydraulic_Press | t          |             16 | -6.6217
 PART-00576 | spindle Hydraulic_Press | t          |             12 | -6.6217
 PART-00327 | spindle Hydraulic_Press | f          |             12 | -6.1783
(10 rows)


## 2. EXPLAIN ANALYZE — proves the query is served by the Lakebase Search index
The plan shows `Index Scan using idx_parts_bm25` with real runtime — the retrieval executed against the index.

In [1]:
plan = conn.execute('''
EXPLAIN ANALYZE SELECT part_id FROM app.parts
 WHERE (search_tsv <@> to_bm25query(to_tsvector('english','spindle'),'app.idx_parts_bm25')) < 0
 ORDER BY (search_tsv <@> to_bm25query(to_tsvector('english','spindle'),'app.idx_parts_bm25')) ASC LIMIT 10;''').fetchall()
print('\n'.join(r[0] for r in plan))


                                                          QUERY PLAN                                                          
------------------------------------------------------------------------------------------------------------------------------
 Limit  (cost=0.00..4.02 rows=1 width=19) (actual time=0.112..0.166 rows=10 loops=1)
   ->  Index Scan using idx_parts_bm25 on parts  (cost=0.00..4.02 rows=1 width=19) (actual time=0.111..0.164 rows=10 loops=1)
         Order By: (search_tsv <@> '(''spindl'':1,app.idx_parts_bm25)'::bm25query_tsvector)
         Filter: ((search_tsv <@> '(''spindl'':1,app.idx_parts_bm25)'::bm25query_tsvector) < '0'::double precision)
 Planning Time: 1.241 ms
 Execution Time: 0.244 ms
(6 rows)


## Result
Retrieval runs **inside the governed Lakebase** via the `lakebase_bm25` index — no separate store. Top candidate: `PART-00031` (spindle Hydraulic_Press, local, 1-day lead).